# SafeCityAI — YOLOv5s traffic detector

This Colab workflow fine-tunes COCO-pretrained YOLOv5s for `Helmet`, `No_Helmet`, and `License_Plate`, validates on a held-out split, plots losses/mAP, runs video inference, and exports ONNX for the API.

**Required before training:** upload an annotated YOLO dataset ZIP with `images/train`, `images/val` (or `images/valid`), `labels/train`, and `labels/val` (or `labels/valid`). Keep frames from the same source recording in one split only. This repository currently has no annotated dataset, so training cannot complete until that ZIP is supplied.

In [ ]:
from pathlib import Path
import os
import shutil
import zipfile

ROOT = Path('/content/safecityai')
if not (ROOT / 'training/train_yolov5.py').exists():
    !git clone https://github.com/akshaydip11-source/object-detection-yolov5-traffic.git {ROOT}
%cd /content/safecityai
!pip install -q -r requirements.txt -r yolov5/requirements.txt
print('GPU:', !nvidia-smi --query-gpu=name --format=csv,noheader)

## 1. Upload and place the labeled dataset

The archive should contain paired images and YOLO `.txt` labels in the folder structure shown above. Each label row is `class_id x_center y_center width height`; coordinates must be normalized to 0–1. The class IDs are 0=Helmet, 1=No_Helmet, 2=License_Plate.

In [ ]:
from google.colab import files
uploaded = files.upload()  # choose the annotated dataset .zip
zip_name = next((name for name in uploaded if name.lower().endswith('.zip')), None)
if not zip_name:
    raise ValueError('Upload a dataset ZIP file')
staging = ROOT / '_dataset_upload'
shutil.rmtree(staging, ignore_errors=True)
staging.mkdir(parents=True)
with zipfile.ZipFile(zip_name) as archive:
    archive.extractall(staging)

def first_dir(paths):
    return next((p for p in paths if p.is_dir()), None)

dataset_roots = [staging] + [p for p in staging.rglob('dataset') if p.is_dir()]
found = None
for base in dataset_roots:
    train_images = first_dir([base/'images/train', base/'train/images'])
    val_images = first_dir([base/'images/val', base/'images/valid', base/'val/images', base/'valid/images'])
    train_labels = first_dir([base/'labels/train', base/'train/labels'])
    val_labels = first_dir([base/'labels/val', base/'labels/valid', base/'val/labels', base/'valid/labels'])
    if all((train_images, val_images, train_labels, val_labels)):
        found = (train_images, val_images, train_labels, val_labels)
        break
if not found:
    raise ValueError('ZIP must contain train and val/valid images and labels folders')

train_images, val_images, train_labels, val_labels = found
for split, source_images, source_labels in [('train', train_images, train_labels), ('val', val_images, val_labels)]:
    target_images = ROOT / 'dataset/images' / split
    target_labels = ROOT / 'dataset/labels' / split
    shutil.rmtree(target_images, ignore_errors=True)
    shutil.rmtree(target_labels, ignore_errors=True)
    shutil.copytree(source_images, target_images)
    shutil.copytree(source_labels, target_labels)
print('Dataset copied into', ROOT / 'dataset')

In [ ]:
# Fail early if the dataset has missing labels, invalid boxes, or missing classes.
!python training/train_yolov5.py --validate-only

## 2. Fine-tune and evaluate

This may take hours depending on the Colab GPU and dataset size. Reduce batch size if CUDA runs out of memory. Training uses the vendored YOLOv5s pipeline, COCO-pretrained weights, mosaic-capable hyperparameters, and a separate validation pass.

In [ ]:
!python training/train_yolov5.py --epochs 50 --batch 16

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

train_runs = sorted((ROOT/'runs/train').glob('safecity-yolov5-*'), key=lambda p: p.stat().st_mtime)
run_dir = train_runs[-1]
results = pd.read_csv(run_dir/'results.csv')
results.columns = results.columns.str.strip()
loss_cols = [c for c in results.columns if c.startswith('train/') and c.endswith('_loss')]
metric_cols = [c for c in results.columns if c.startswith('metrics/') or 'mAP' in c]
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
results[loss_cols].plot(ax=axes[0], title='Training losses')
results[metric_cols].plot(ax=axes[1], title='Validation metrics (including mAP@0.5)')
plt.tight_layout(); plt.show()
print('Training run:', run_dir)
print('Validation summary:')
print((ROOT/'runs/val'/run_dir.name/'results.txt').read_text())

## 3. Run inference on a held-out street video

Use a video that was not used to create training or validation frames. Inspect false positives and missed detections before using the model for any enforcement workflow.

In [ ]:
video_upload = files.upload()  # choose the held-out .mp4/.avi test clip
video_name = next((name for name in video_upload if Path(name).suffix.lower() in {'.mp4','.avi','.mov','.mkv','.webm'}), None)
if not video_name:
    raise ValueError('Upload a supported video file')
best = run_dir/'weights/best.pt'
!python yolov5/detect.py --weights {best} --source {video_name} --img 640 --conf-thres 0.5 --save-txt --save-conf --project outputs --name safecityai-demo --exist-ok
from IPython.display import Video, display
result_video = next((ROOT/'outputs/safecityai-demo').glob('*.mp4'))
display(Video(str(result_video), embed=True))

## 4. Export the model artifact for the FastAPI service

The training script already exports ONNX into `backend/weights/yolov5_custom.onnx` and writes `backend/weights/traffic.names`. Download both artifacts, add them to the deployment source, and configure Render with `MODEL_PATH=weights/yolov5_custom.onnx` and `CLASS_NAMES_PATH=weights/traffic.names`. Do not label the deployed model as custom until its health endpoint reports that model and an image/video inference has been reviewed.

In [ ]:
files.download(str(ROOT/'backend/weights/yolov5_custom.onnx'))
files.download(str(ROOT/'backend/weights/traffic.names'))